# Task 3 — GenAI Summarization

## Goal

Generate concise summaries for the final review clusters discovered in Task 2.

Each summary should describe the main themes, common customer feedback, and notable sentiment patterns inside a cluster.

The summaries will be based on the clustered review text and the final human-readable cluster interpretation.

## Daily Recap

### Previous work

Task 2 — Clustering is complete.

We tested KMeans with K = 4, 5 and 6 using TF-IDF features from `reviews.text`.

K = 6 was selected because it achieved the highest Silhouette Score among the tested values and produced the clearest semantic separation.

Four cluster themes are already aligned between both team analyses:

- Kindle / Reading
- Ease of Use / Setup
- Kids / Family / Gifts
- Tablets / Value

At the start of Task 3, two broader cluster names still required team alignment; the final six-theme mapping is recorded in the validated Task 2 results and must be checked against the clustered dataset before any relabelling.

### Today

We begin Task 3 — GenAI Summarization.

The goal is to create one concise summary per final cluster based on real customer reviews.

CUSTOMER REVIEWS
        │
        ├──── sentiment model
        │        ↓
        │   positive / neutral / negative
        │
        ├──── clustering
        │        ↓
        │   recurring topics
        │
        └──── GenAI summarization
                 ↓
           business insight

In [16]:
# Install the libraries required to run T5 locally.
%pip install torch transformers sentencepiece

Note: you may need to restart the kernel to use updated packages.


# Task 3 — Generative Summarization

## 1. Load the clustered dataset

### Question

What data from Task 2 do we need for summarization?

### What we did

We loaded `clustered_reviews.csv`, which contains the cleaned reviews together with:

- sentiment labels
- numerical cluster IDs
- human-readable cluster names

### Result

The dataset contains 34,615 reviews and the six final clusters from Task 2.

### Decision / Finding

Task 3 will use the clusters discovered in Task 2 as the basis for generating summaries.

In [17]:
# Import pandas for working with the clustered review data.
import pandas as pd

# Import Path to build reliable file paths.
from pathlib import Path

# Find the project root folder.
PROJECT_ROOT = Path.cwd().resolve()

# If the notebook is running from the notebooks folder,
# move one level up to the project root.
if not (PROJECT_ROOT / "datasets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Build the path to the clustered dataset from Task 2.
CLUSTERED_DATA_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "clustered_reviews.csv"
)

# Load the clustered reviews.
df = pd.read_csv(
    CLUSTERED_DATA_PATH,
    low_memory=False
)

# Check the dataset dimensions.
print("Dataset shape:", df.shape)

# Confirm that the clustering columns exist.
assert "cluster" in df.columns
assert "cluster_name" in df.columns

print("Clustered dataset loaded successfully.")

Dataset shape: (34615, 9)
Clustered dataset loaded successfully.


In [18]:
# Update the cluster names to the final
# semantic names agreed by the team.
cluster_names = {
    0: "Kindle & Reading",
    1: "Ease of Use & Setup",
    2: "Fire TV & Streaming",
    3: "General & Mixed Feedback",
    4: "Kids, Family & Gifts",
    5: "Tablets & Value"
}

# Replace the old readable names using the cluster ID.
df["cluster_name"] = df["cluster"].map(cluster_names)

# Check that the final names are now correct.
df[
    ["cluster", "cluster_name"]
].drop_duplicates().sort_values("cluster")

,cluster,cluster_name
6,0,Kindle & Reading
7,1,Ease of Use & Setup
0,2,Fire TV & Streaming
32,3,General & Mixed Feedback
1,4,"Kids, Family & Gifts"
2,5,Tablets & Value


## 1. Prepare representative review samples

### Question

How should we choose reviews for GenAI summarization?

### Decision

We use sentiment-aware sampling instead of purely random sampling.

The original dataset is strongly skewed toward Positive reviews. A random sample could therefore miss important Neutral and Negative feedback.

For each cluster, we aim to sample:

- 20 Positive reviews
- 6 Neutral reviews
- 4 Negative reviews

This keeps the sample mostly representative of the dataset while ensuring minority feedback is visible to the summarization model.

In [19]:
# Number of reviews we want from each sentiment group.
POSITIVE_SAMPLE = 20
NEUTRAL_SAMPLE = 6
NEGATIVE_SAMPLE = 4

# Create an empty dictionary.
# Each cluster ID will store a small representative sample.
cluster_samples = {}

# Loop through every cluster ID in sorted order.
for cluster_id in sorted(df["cluster"].unique()):

    # Select only reviews from the current cluster.
    cluster_df = df[
        df["cluster"] == cluster_id
    ]

    # Select Positive reviews from this cluster.
    positive_reviews = cluster_df[
        cluster_df["sentiment"] == "positive"
    ]

    # Select Neutral reviews from this cluster.
    neutral_reviews = cluster_df[
        cluster_df["sentiment"] == "neutral"
    ]

    # Select Negative reviews from this cluster.
    negative_reviews = cluster_df[
        cluster_df["sentiment"] == "negative"
    ]

    # Sample Positive reviews.
    positive_sample = positive_reviews.sample(
        n=min(POSITIVE_SAMPLE, len(positive_reviews)),
        random_state=42
    )

    # Sample Neutral reviews.
    neutral_sample = neutral_reviews.sample(
        n=min(NEUTRAL_SAMPLE, len(neutral_reviews)),
        random_state=42
    )

    # Sample Negative reviews.
    negative_sample = negative_reviews.sample(
        n=min(NEGATIVE_SAMPLE, len(negative_reviews)),
        random_state=42
    )

    # Combine the three sentiment groups.
    representative_sample = pd.concat(
        [
            positive_sample,
            neutral_sample,
            negative_sample
        ]
    )

    # Store only the review text in the dictionary.
    cluster_samples[cluster_id] = (
        representative_sample["reviews.text"]
        .tolist()
    )

# Check how many reviews were stored for each cluster.
for cluster_id, reviews in cluster_samples.items():
    print(
        f"Cluster {cluster_id}:",
        len(reviews),
        "reviews"
    )

Cluster 0: 30 reviews
Cluster 1: 30 reviews
Cluster 2: 30 reviews
Cluster 3: 30 reviews
Cluster 4: 30 reviews
Cluster 5: 30 reviews


## 2. Build the summarization prompt

### Question

How do we give the LLM enough context to summarize a cluster correctly?

### What we do

For each cluster, we provide:

- the cluster name
- a representative sample of reviews
- explicit instructions about what information to extract

The goal is to reduce vague outputs and avoid hallucinated details.

In [20]:
# Create a function that builds one prompt
# for one review cluster.
def build_cluster_prompt(
    cluster_name,
    reviews
):
    # Convert the list of reviews into one
    # numbered block of text.
    reviews_text = ""

    for index, review in enumerate(
        reviews,
        start=1
    ):
        reviews_text += (
            f"{index}. {review}\n"
        )

    # Build the final prompt.
    prompt = f"""
You are analyzing customer reviews.

Cluster:
{cluster_name}

Below is a representative sample of reviews from this cluster.

Reviews:
{reviews_text}

Your task:

1. Identify the main themes in the reviews.
2. Describe the most common positive feedback.
3. Describe the most common complaints or concerns.
4. Write a short business-oriented summary of what customers are saying.
5. Use only information supported by the reviews.
6. Do not invent details that are not present in the reviews.

Keep the response concise and clear.
"""

    return prompt

In [21]:
# Select one cluster ID for testing.
test_cluster_id = 0

# Find the readable name for that cluster.
test_cluster_name = (
    df[
        df["cluster"] == test_cluster_id
    ]["cluster_name"]
    .iloc[0]
)

# Build the prompt for this test cluster.
test_prompt = build_cluster_prompt(
    test_cluster_name,
    cluster_samples[test_cluster_id]
)

# Print only the first part of the prompt
# so we can inspect it before sending it to an LLM.
print(
    test_prompt[:3000]
)


You are analyzing customer reviews.

Cluster:
Kindle & Reading

Below is a representative sample of reviews from this cluster.

Reviews:
1. This e-reader is amazing. It is so light and easy to read with one hand. The screen is great and the battery lasts for weeks.
2. With a new higher resolution display, Bookerly font, and long lasting battery, the Kindle Paperwhite is the best e-readr in the market today. Reads like real paper. The device capabilities are complemented by Kindle's vast e-book store.
3. Perfect products with perfect recommendations. I enjoy with kindle. Thank you
4. I just love the size of the kindle and it's easy to take on the go. Perfect reading for outdoors on the deck around the pool.
5. This is great I was used to just the plain Kindle and this feels more like a tablet than just a kindle. I use it to check email, do pintrest and read my books. Totally in love with my new Kindle fire!
6. I've always been a big proponent of real books; love the smell, touch, etc, 

# 2. Create a balanced qualitative sample

### Question

How should we select reviews for testing the generative model?

### What we did

For each cluster, we selected:

- 5 Positive reviews
- 5 Neutral reviews
- 5 Negative reviews

This creates 15 reviews per cluster and 90 reviews in total.

### Result

Each of the six clusters contains an equal qualitative sample of the three sentiment groups.

### Decision / Finding

This balanced sample is used only to evaluate the quality of the generated summaries.

It must not be used to estimate the real sentiment proportions of a cluster. Real sentiment statistics come from the full dataset.

In [25]:
# Create a balanced qualitative sample:
# 5 positive, 5 neutral and 5 negative reviews
# from each cluster.

samples = []

for cluster_id in sorted(df["cluster"].unique()):

    # Select only reviews from the current cluster.
    cluster_df = df[
        df["cluster"] == cluster_id
    ]

    # Go through each sentiment category.
    for sentiment in [
        "positive",
        "neutral",
        "negative"
    ]:

        # Select only reviews with this sentiment.
        sentiment_df = cluster_df[
            cluster_df["sentiment"] == sentiment
        ]

        # Take up to 5 reviews.
        n = min(
            5,
            len(sentiment_df)
        )

        # Only sample if reviews exist.
        if n > 0:
            sample = sentiment_df.sample(
                n=n,
                random_state=42
            )

            samples.append(sample)

# Combine all sampled reviews into one DataFrame.
summary_sample = pd.concat(
    samples,
    ignore_index=True
)

# Check the result.
print(
    "Balanced qualitative sample shape:",
    summary_sample.shape
)

summary_sample[
    ["cluster", "cluster_name", "sentiment"]
].value_counts().sort_index()

Balanced qualitative sample shape: (90, 9)


cluster  cluster_name              sentiment
0        Kindle & Reading          negative     5
                                   neutral      5
                                   positive     5
1        Ease of Use & Setup       negative     5
                                   neutral      5
                                   positive     5
2        Fire TV & Streaming       negative     5
                                   neutral      5
                                   positive     5
3        General & Mixed Feedback  negative     5
                                   neutral      5
                                   positive     5
4        Kids, Family & Gifts      negative     5
                                   neutral      5
                                   positive     5
5        Tablets & Value           negative     5
                                   neutral      5
                                   positive     5
Name: count, dtype: int64

## 3. Prepare the local generative-model environment

### Question

Can we run a pretrained generative model locally without using an external API?

### What we did

We installed and verified:

- PyTorch
- Hugging Face Transformers
- SentencePiece

We also checked whether CUDA GPU acceleration is available.

### Result

PyTorch and Transformers are working correctly.

CUDA is not available, so the model will run on CPU.

### Decision / Finding

CPU is sufficient for this first baseline because we are only performing inference with a small pretrained model, not training or fine-tuning it.

## 3. Prepare the local generative-model environment

### Question

Can we run a pretrained generative model locally without using an external API?

### What we did

We installed and verified:

- PyTorch
- Hugging Face Transformers
- SentencePiece

We also checked whether CUDA GPU acceleration is available.

### Result

PyTorch and Transformers are working correctly.

CUDA is not available, so the model will run on CPU.

### Decision / Finding

CPU is sufficient for this first baseline because we are only performing inference with a small pretrained model, not training or fine-tuning it.

In [22]:
# Check which Python environment is running
# and whether the required libraries are available.
import sys
import torch
import transformers

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

Python: /Users/caiomaia/ironhack-workspace/.venv/bin/python3.12.real
PyTorch: 2.14.1
Transformers: 5.18.0
CUDA available: False


## 4. Load the pretrained T5-small model

### Question

Which pretrained generative model will we use for the first summarization baseline?

### What we did

We loaded `google-t5/t5-small` using Hugging Face Transformers.

T5 is a sequence-to-sequence model: it receives text as input and generates text as output.

### Result

The tokenizer and pretrained T5-small model loaded successfully.

### Decision / Finding

T5-small will be our first local summarization baseline.

No fine-tuning is used at this stage.

In [23]:
# Import the tokenizer and the correct model class
# for T5, which is a sequence-to-sequence model.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Name of the pretrained model on Hugging Face.
model_name = "google-t5/t5-small"

# Load the tokenizer.
# The tokenizer converts text into numbers the model can understand.
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the pretrained T5-small model.
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("T5 model loaded successfully!")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 12805.69it/s]

T5 model loaded successfully!


## 5. Validate basic T5 inference

### Question

Can T5-small successfully generate a summary before we use real customer reviews?

### What we did

We gave the model a short synthetic Kindle example containing both positive feedback and complaints.

### Result

T5 produced a valid summary and preserved both the positive and negative information without inventing new facts.

However, the output was close to the original wording and showed limited abstraction.

### Decision / Finding

The inference pipeline works correctly.

T5-small is usable as a baseline, but its summarization quality still needs to be evaluated on real review data.

In [24]:
# Create a very small test input first.
# T5 expects the instruction "summarize:" before the text.
test_text = """
summarize: Customers like the Kindle because it is lightweight,
easy to use and convenient for reading at night.
Some customers complain about advertisements,
limited app availability and occasional software problems.
"""

# Convert the text into tokens that T5 can process.
inputs = tokenizer(
    test_text,
    return_tensors="pt",
    max_length=512,
    truncation=True
)

# Ask T5 to generate a summary.
outputs = model.generate(
    **inputs,
    max_new_tokens=60,
    min_new_tokens=15,
    num_beams=4,
    early_stopping=True
)

# Convert the generated tokens back into readable text.
summary = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(summary)

the Kindle is lightweight, easy to use and convenient for reading at night. some customers complain about advertisements, limited app availability and occasional software problems.


## 6. Test T5 on one real cluster

### Question

Can T5-small produce a useful summary from real reviews in the `Kindle & Reading` cluster?

### What we did

We selected the 15 balanced Kindle reviews:

- 5 Positive
- 5 Neutral
- 5 Negative

Before generating a summary, we inspected the full review text and measured the model input length.

In [26]:
# Select only reviews from the Kindle & Reading cluster.
kindle_df = summary_sample[
    summary_sample["cluster_name"] == "Kindle & Reading"
]

# Show the sentiment and review text
# so we know exactly what T5 will receive.
kindle_df[
    ["sentiment", "reviews.text"]
].reset_index(drop=True)

,sentiment,reviews.text
0,positive,This e-reader is amazing. It is so light and e...
1,positive,"With a new higher resolution display, Bookerly..."
2,positive,Perfect products with perfect recommendations....
3,positive,I just love the size of the kindle and it's ea...
4,positive,This is great I was used to just the plain Kin...
5,neutral,I had to replace my Kindle 3G keyboard after s...
6,neutral,Its a decent reader for beginners. There is no...
7,neutral,The Touch cover from Amazon has some good and ...
8,neutral,I got it to store all my college books without...
9,neutral,"Everyone knows ""kindle fire"". When these went ..."


In [27]:
# Print every Kindle review in full.
# This lets us inspect the exact evidence
# that will later be given to T5.

for index, row in kindle_df.reset_index(drop=True).iterrows():

    # Print the review number and sentiment.
    print(
        f"Review {index + 1} "
        f"({row['sentiment']}):"
    )

    # Print the complete review text.
    print(row["reviews.text"])

    # Add a separator to make the reviews easier to read.
    print("-" * 80)

Review 1 (positive):
This e-reader is amazing. It is so light and easy to read with one hand. The screen is great and the battery lasts for weeks.
--------------------------------------------------------------------------------
Review 2 (positive):
With a new higher resolution display, Bookerly font, and long lasting battery, the Kindle Paperwhite is the best e-readr in the market today. Reads like real paper. The device capabilities are complemented by Kindle's vast e-book store.
--------------------------------------------------------------------------------
Review 3 (positive):
Perfect products with perfect recommendations. I enjoy with kindle. Thank you
--------------------------------------------------------------------------------
Review 4 (positive):
I just love the size of the kindle and it's easy to take on the go. Perfect reading for outdoors on the deck around the pool.
--------------------------------------------------------------------------------
Review 5 (positive):
This

In [28]:
# Combine the 15 Kindle reviews into one text block.
kindle_reviews_text = "\n".join(
    kindle_df["reviews.text"].astype(str)
)

# Add the T5 summarization instruction.
kindle_input = (
    "summarize: "
    + kindle_reviews_text
)

# Tokenize without truncating so we can measure
# how large the complete input actually is.
kindle_tokens = tokenizer(
    kindle_input,
    truncation=False
)

# Count the number of input tokens.
token_count = len(
    kindle_tokens["input_ids"]
)

print("Kindle input token count:", token_count)
print("Current T5 input limit used in our code: 512")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1076 > 512). Running this sequence through the model will result in indexing errors


Kindle input token count: 1076
Current T5 input limit used in our code: 512


INPUT
All 15 real Kindle reviews.
TRANSFORMATION
The tokenizer converts the complete text into T5 tokens, but this time we do not cut anything.
OUTPUT
Something like:
Kindle input token count: 900
Current T5 input limit used in our code: 512

WHY
Before generating a summary, we need to know whether T5 can actually see all the evidence we selected.
This is an important AI-engineering lesson:
A balanced dataset sample is useless if the model's context window silently removes part of it.

Run that token-count cell and send me the number. Then we'll decide how to construct the T5 input without losing the balance.

In [29]:
# Measure the token size of each sentiment group
# inside the Kindle & Reading sample.

for sentiment in ["positive", "neutral", "negative"]:

    sentiment_text = "\n".join(
        kindle_df[
            kindle_df["sentiment"] == sentiment
        ]["reviews.text"].astype(str)
    )

    sentiment_input = (
        "summarize: "
        + sentiment_text
    )

    sentiment_tokens = tokenizer(
        sentiment_input,
        truncation=False
    )

    token_count = len(
        sentiment_tokens["input_ids"]
    )

    print(
        sentiment,
        "token count:",
        token_count
    )

positive token count: 195
neutral token count: 731
negative token count: 156


In [30]:
# Measure the token length of each neutral Kindle review.

neutral_reviews = kindle_df[
    kindle_df["sentiment"] == "neutral"
].reset_index(drop=True)

for index, review in neutral_reviews["reviews.text"].items():

    review_input = (
        "summarize: "
        + str(review)
    )

    review_tokens = tokenizer(
        review_input,
        truncation=False
    )

    token_count = len(
        review_tokens["input_ids"]
    )

    print(
        f"Neutral review {index + 1}:",
        token_count,
        "tokens"
    )

Neutral review 1: 174 tokens
Neutral review 2: 43 tokens
Neutral review 3: 361 tokens
Neutral review 4: 81 tokens
Neutral review 5: 84 tokens


In [31]:
# Combine the 5 positive Kindle reviews.
positive_text = "\n".join(
    kindle_df[
        kindle_df["sentiment"] == "positive"
    ]["reviews.text"].astype(str)
)

# Add the T5 task instruction.
positive_input = (
    "summarize: "
    + positive_text
)

# Convert the text into PyTorch tensors.
inputs = tokenizer(
    positive_input,
    return_tensors="pt",
    max_length=512,
    truncation=True
)

# Generate the summary.
outputs = model.generate(
    **inputs,
    max_new_tokens=60,
    min_new_tokens=15,
    num_beams=4,
    early_stopping=True
)

# Convert generated tokens back into readable text.
positive_summary = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(positive_summary)

the Kindle Paperwhite is the best e-readr in the market today. it is so light and easy to read with one hand. the screen is great and the battery lasts for weeks.


### Result

The complete Kindle sample contains 1,076 tokens.

The current T5-small input limit is 512 tokens.

The sentiment groups contain:

- Positive: 195 tokens
- Neutral: 731 tokens
- Negative: 156 tokens

The Neutral group exceeds the model input limit because some individual reviews are much longer than the others.

### Decision / Finding

We should not concatenate all 15 reviews and rely on truncation, because important evidence would be discarded.

Instead, we will test smaller chunks and evaluate their summaries before building a full multi-step summarization process.

## 7. Compare pretrained generative baselines

### Question

Which pretrained generative model produces the most useful and faithful multi-review summary?

### Evaluation criteria

We compare models using the same Kindle & Reading sample and evaluate:

- synthesis quality
- coverage of strengths
- coverage of complaints
- information retention
- hallucination risk
- ability to produce a short consumer-review article
- suitability for the MVP

### Current team observations

| Model | Observation |
|---|---|
| T5-small | Very extractive; weak aggregation |
| FLAN-T5-small | Better instruction following, but still extractive |
| DistilBART | Better aggregation, but loses information |
| Qwen2.5-3B-Instruct | Best candidate so far; final validation pending |

## 8. Prepare standardized multi-review inputs

### Question

How can we ensure that every model receives the same review evidence?

### What we do

We convert the balanced qualitative sample into one standardized text block per cluster.

Each block contains:

- the cluster name
- 5 Positive reviews
- 5 Neutral reviews
- 5 Negative reviews
- the sentiment label for each review

### Decision

The review evidence will remain fixed while we compare different generative models.

This makes the model comparison fairer because only the model and generation strategy change.

In [32]:
# Create an empty dictionary.
# Each cluster name will store one standardized review block.
cluster_review_blocks = {}

# Go through each final cluster name.
for cluster_name in sorted(summary_sample["cluster_name"].unique()):

    # Select the 15 balanced reviews for this cluster.
    cluster_reviews = summary_sample[
        summary_sample["cluster_name"] == cluster_name
    ].reset_index(drop=True)

    # Start an empty list where each formatted review will be stored.
    formatted_reviews = []

    # Go through the reviews one by one.
    for index, row in cluster_reviews.iterrows():

        # Create a readable line containing:
        # review number, sentiment, and full review text.
        formatted_review = (
            f"Review {index + 1} "
            f"[{row['sentiment']}]: "
            f"{row['reviews.text']}"
        )

        formatted_reviews.append(formatted_review)

    # Join all 15 reviews into one text block.
    cluster_review_blocks[cluster_name] = "\n\n".join(
        formatted_reviews
    )

# Inspect only the Kindle block for now.
print(
    cluster_review_blocks["Kindle & Reading"]
)

Review 1 [positive]: This e-reader is amazing. It is so light and easy to read with one hand. The screen is great and the battery lasts for weeks.

Review 2 [positive]: With a new higher resolution display, Bookerly font, and long lasting battery, the Kindle Paperwhite is the best e-readr in the market today. Reads like real paper. The device capabilities are complemented by Kindle's vast e-book store.

Review 3 [positive]: Perfect products with perfect recommendations. I enjoy with kindle. Thank you

Review 4 [positive]: I just love the size of the kindle and it's easy to take on the go. Perfect reading for outdoors on the deck around the pool.

Review 5 [positive]: This is great I was used to just the plain Kindle and this feels more like a tablet than just a kindle. I use it to check email, do pintrest and read my books. Totally in love with my new Kindle fire!

Review 6 [neutral]: I had to replace my Kindle 3G keyboard after several years because it would no longer charge. So I got

## Final Production Model Decision

The local model experiments showed that **Qwen2.5-3B-Instruct**
was the strongest open-source model tested in this project.

Compared with T5-small, FLAN-T5-small and DistilBART, Qwen produced
better multi-review synthesis and followed the grounded
consumer-insight prompt more effectively.

We then evaluated **NVIDIA Nemotron 3 Ultra 550B** using the same
Kindle & Reading review sample, cluster statistics and grounded prompt.

Nemotron produced a more detailed and structured synthesis and can be
accessed through a hosted API without requiring the project to host a
large language model directly.

Following the deployment architecture discussed with the instructor,
the project therefore uses:

- **Qwen2.5-3B-Instruct** as the strongest local/open-source baseline.
- **NVIDIA Nemotron 3 Ultra 550B API** as the production GenAI
  summarization model.

Nemotron still produced some unsupported generalizations, showing that
a larger model does not eliminate hallucination.

Grounded prompts, explicit evidence constraints and output validation
remain necessary.

### Final Production Summarization Pipeline

    Cluster
       ↓
    Calculate real cluster statistics
       ↓
    Balanced qualitative sample
    (5 positive + 5 neutral + 5 negative)
       ↓
    Grounded Prompt V3
       ↓
    NVIDIA Nemotron API
       ↓
    Consumer Insight Article

The balanced sample is used to expose different types of customer
feedback, while the full cluster statistics are used to describe the
actual sentiment distribution.

## 9. Production Summarization with NVIDIA Nemotron

### Question

Can the instructor-recommended NVIDIA Nemotron API produce grounded
consumer-insight summaries for all six review clusters?

### What we do

We reuse the same methodology developed during the local model experiments:

- the same six clusters
- real cluster-level statistics
- balanced qualitative sampling
- Grounded Prompt V3

The main change is the generative model.

Instead of running a large model locally, the final production pipeline
calls NVIDIA Nemotron through a hosted API.

### Decision

Qwen2.5-3B-Instruct remains the strongest local/open-source baseline.

NVIDIA Nemotron is evaluated as the production summarization model
because it provides stronger synthesis without requiring us to host
a large language model ourselves.

In [34]:
# Install python-dotenv into the current Jupyter environment.
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [36]:
# API credentials are loaded in the next cell, after importing os and python-dotenv.

In [42]:
# Import os so Python can read environment variables.
import os

# Load values stored in the local .env file.
from dotenv import load_dotenv

# Load the .env file into the current Python environment.
load_dotenv()

# Read the NVIDIA API key.
nvidia_api_key = os.getenv("NVIDIA_API_KEY")

# Check only whether a key was found.
# Never print the actual key.
if not nvidia_api_key:
    raise RuntimeError("NVIDIA_API_KEY was not found. Add it to the local .env file before calling the NVIDIA API.")

print("NVIDIA API key loaded successfully.")

NVIDIA API key loaded successfully.


## 10. Validate the NVIDIA Nemotron API connection

### Question

Can the notebook successfully call the NVIDIA-hosted Nemotron model?

### What we do

We send one very small test prompt through the NVIDIA API.

This test checks:

- API authentication
- endpoint connectivity
- model availability
- response parsing

### Decision

We validate the API with a simple request before sending real customer-review data.

In [41]:
# Install the OpenAI Python client into the current Jupyter environment.
%pip install openai

  Using cached httpx2-2.13.1-py3-none-any.whl.metadata (9.8 kB)
  Using cached httpcore2-2.13.1-py3-none-any.whl.metadata (26 kB)
  Using cached truststore-0.10.4-py3-none-any.whl.metadata (4.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 3.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 5.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [openai] 9/10 [openai]c]
Note: you may need to restart the kernel to use updated packages.


In [43]:
# Import the OpenAI-compatible client.
# NVIDIA's hosted endpoint uses an OpenAI-compatible API format.
from openai import OpenAI

# Create a client that points to NVIDIA's API instead of OpenAI's API.
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=nvidia_api_key
)

# Send one small test request.
response = client.chat.completions.create(
    model="nvidia/nemotron-3-ultra-550b-a55b",
    messages=[
        {
            "role": "user",
            "content": (
                "Summarize this in one short sentence: "
                "Customers like the Kindle because it is lightweight "
                "and easy to read, but some complain about software issues."
            )
        }
    ],
    temperature=0.2,
    max_tokens=100
)

# Print only the model's final text response.
print(response.choices[0].message.content)

Customers appreciate the Kindle's lightweight design and readability, though some report software problems.


## 11. Test Nemotron on a real review cluster

### Question

Can Nemotron produce a grounded consumer-insight summary from the real
Kindle & Reading review sample?

### What we do

We reuse:

- the same 15 balanced Kindle reviews
- the same real cluster statistics
- the same Grounded Prompt V3 used during model comparison

This keeps the comparison fair and allows us to evaluate whether the API
model improves synthesis without changing the underlying evidence.

### Decision

We test one real cluster first before automating generation for all six clusters.

## 12. Generalize Nemotron summarization to all clusters

### Question

Can we reuse the same grounded summarization pipeline for all six clusters?

### What we do

We keep the same:

- full-cluster statistics
- balanced 5/5/5 qualitative sample
- Grounded Prompt V3
- fixed four-section article structure

The only change is the generation backend:

**Qwen local model → NVIDIA Nemotron API**

### Decision

If the six outputs remain grounded and useful, Nemotron becomes the production summarization model.

In [44]:
def generate_cluster_summary_nemotron(
    cluster_name,
    df,
    client
):
    # 1. Select the full cluster.
    cluster_df = df[
        df["cluster_name"] == cluster_name
    ].copy()

    # 2. Calculate real statistics from ALL reviews
    # in this cluster.
    total_reviews = len(cluster_df)

    avg_rating = (
        cluster_df["reviews.rating"]
        .mean()
    )

    sentiment_pct = (
        cluster_df["sentiment"]
        .value_counts(normalize=True)
        .mul(100)
    )

    # 3. Create a balanced qualitative sample:
    # up to 5 positive, 5 neutral and 5 negative reviews.
    samples = []

    for sentiment in [
        "positive",
        "neutral",
        "negative"
    ]:
        sentiment_df = cluster_df[
            cluster_df["sentiment"] == sentiment
        ]

        n = min(
            5,
            len(sentiment_df)
        )

        if n > 0:
            sample = sentiment_df.sample(
                n=n,
                random_state=42
            )

            samples.append(sample)

    # Combine the sampled reviews.
    sample_df = pd.concat(
        samples,
        ignore_index=True
    )

    # 4. Convert the sampled reviews into
    # one readable text block.
    reviews_text = ""

    for i, row in sample_df.iterrows():
        reviews_text += (
            f"\nReview {i + 1} "
            f"(Rating: {row['reviews.rating']}/5, "
            f"Sentiment: {row['sentiment']}):\n"
            f"{row['reviews.text']}\n"
        )

    # 5. Build the same grounded Prompt V3.
    prompt = f"""
You are a customer review analyst.

Analyze customer feedback for the category: "{cluster_name}".

REAL CLUSTER STATISTICS:
- Total reviews: {total_reviews}
- Average rating: {avg_rating:.2f}/5
- Positive sentiment: {sentiment_pct.get('positive', 0):.1f}%
- Neutral sentiment: {sentiment_pct.get('neutral', 0):.1f}%
- Negative sentiment: {sentiment_pct.get('negative', 0):.1f}%

SAMPLING NOTE:
The reviews below were deliberately balanced across positive,
neutral and negative sentiment whenever enough reviews were available.

This sample is designed to expose strengths and problems.
It does NOT represent the real sentiment distribution.
Use the real statistics above for overall customer sentiment.

STRICT GROUNDING RULES:
- Use ONLY information explicitly present in the reviews or statistics.
- Do NOT infer additional product problems or features.
- Do NOT call something "common", "frequent" or "widespread"
  based on this small sample.
- If only one review mentions an issue, present it as an example,
  not as a general pattern.
- Do NOT invent information about customer support, pricing,
  storage, compatibility or features.
- Star ratings and text may disagree; prioritize the review text.
- Synthesize the evidence rather than copying review sentences.

TASK:
Write a concise consumer insight article of 180-230 words.

Use exactly these sections:

### Overall Customer Perception
Use the real cluster statistics to describe overall satisfaction.

### Strengths & Use Cases
Describe strengths and use cases supported by the review sample.

### Complaints & Trade-offs
Describe only limitations explicitly supported by the reviews.

### Buyer Takeaway
Provide a balanced conclusion based only on the evidence.

REVIEWS:
{reviews_text}
"""

    # 6. Send the grounded prompt to NVIDIA Nemotron.
    completion = client.chat.completions.create(
        model="nvidia/nemotron-3-ultra-550b-a55b",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an evidence-grounded customer review analyst. "
                    "Never add facts that are not explicitly supported "
                    "by the supplied data."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2,
        top_p=0.95,
        max_tokens=600,
        extra_body={
            "chat_template_kwargs": {
                "enable_thinking": False
            }
        },
        stream=False
    )

    # 7. Extract only the generated article.
    summary = (
        completion
        .choices[0]
        .message
        .content
    )

    return summary

In [45]:
# Test the new production function
# on Kindle & Reading before running all six clusters.
kindle_nemotron_summary = generate_cluster_summary_nemotron(
    "Kindle & Reading",
    df,
    client
)

print(kindle_nemotron_summary)

### Overall Customer Perception
The Kindle & Reading category shows strong overall satisfaction, with 4,679 reviews averaging 4.67 out of 5 stars. Positive sentiment dominates at 94.9%, while neutral and negative sentiments account for 3.1% and 2.0% respectively. The sampled reviews, deliberately balanced to surface issues, reveal a mix of enthusiastic endorsements and specific functional frustrations.

### Strengths & Use Cases
Reviewers consistently praise the devices for portability and reading comfort. The lightweight, one-handed design is highlighted as ideal for travel, outdoor reading by the pool, and general on-the-go use. The high-resolution display and Bookerly font are described as reading "like real paper," and battery life lasting weeks is a major advantage. The integrated Kindle store ecosystem provides seamless access to a vast library. Some users utilize Fire tablets for broader functionality, including email, web browsing, and apps like Pinterest.

### Complaints & Tra

In [46]:
# Select the full Kindle & Reading cluster.
kindle_cluster = df[
    df["cluster_name"] == "Kindle & Reading"
].copy()

# Calculate the real statistics using ALL Kindle reviews.
total_reviews = len(kindle_cluster)

avg_rating = (
    kindle_cluster["reviews.rating"]
    .mean()
)

sentiment_pct = (
    kindle_cluster["sentiment"]
    .value_counts(normalize=True)
    .mul(100)
)

# Create the balanced qualitative sample:
# 5 positive, 5 neutral and 5 negative reviews.
kindle_samples = []

for sentiment in ["positive", "neutral", "negative"]:

    sentiment_df = kindle_cluster[
        kindle_cluster["sentiment"] == sentiment
    ]

    n = min(
        5,
        len(sentiment_df)
    )

    if n > 0:
        sample = sentiment_df.sample(
            n=n,
            random_state=42
        )

        kindle_samples.append(sample)

# Combine the sampled reviews.
kindle_sample_df = pd.concat(
    kindle_samples,
    ignore_index=True
)

# Convert the reviews into one readable text block.
reviews_text = ""

for i, row in kindle_sample_df.iterrows():

    reviews_text += (
        f"\nReview {i + 1} "
        f"(Rating: {row['reviews.rating']}/5, "
        f"Sentiment: {row['sentiment']}):\n"
        f"{row['reviews.text']}\n"
    )

# Build Grounded Prompt V3.
prompt_v3 = f"""
You are a customer review analyst.

Analyze customer feedback for the category: "Kindle & Reading".

REAL CLUSTER STATISTICS:
- Total reviews: {total_reviews}
- Average rating: {avg_rating:.2f}/5
- Positive sentiment: {sentiment_pct.get('positive', 0):.1f}%
- Neutral sentiment: {sentiment_pct.get('neutral', 0):.1f}%
- Negative sentiment: {sentiment_pct.get('negative', 0):.1f}%

SAMPLING NOTE:
The reviews below were deliberately balanced across positive,
neutral and negative sentiment.

This sample is designed to expose strengths and problems.
It does NOT represent the real sentiment distribution.
Use the real statistics above for overall customer sentiment.

STRICT GROUNDING RULES:
- Use ONLY information explicitly present in the reviews or statistics.
- Do NOT infer additional product problems or features.
- Do NOT call something "common", "frequent" or "widespread"
  based on this small sample.
- If only one review mentions an issue, present it as an example,
  not as a general pattern.
- Do NOT invent information about customer support, pricing,
  storage, compatibility or features.
- Star ratings and text may disagree; prioritize the review text.
- Synthesize the evidence rather than copying review sentences.

TASK:
Write a concise consumer insight article of 180-230 words.

Use exactly these sections:

### Overall Customer Perception
Use the real cluster statistics to describe overall satisfaction.

### Strengths & Use Cases
Describe strengths and use cases supported by the review sample.

### Complaints & Trade-offs
Describe only limitations explicitly supported by the reviews.

### Buyer Takeaway
Provide a balanced conclusion based only on the evidence.

REVIEWS:
{reviews_text}
"""

# Send the real Kindle evidence to NVIDIA Nemotron.
completion = client.chat.completions.create(
    model="nvidia/nemotron-3-ultra-550b-a55b",
    messages=[
        {
            "role": "system",
            "content": (
                "You are an evidence-grounded customer review analyst. "
                "Never add facts that are not explicitly supported "
                "by the supplied data."
            )
        },
        {
            "role": "user",
            "content": prompt_v3
        }
    ],
    temperature=0.2,
    top_p=0.95,
    max_tokens=600,
    extra_body={
        "chat_template_kwargs": {
            "enable_thinking": False
        }
    },
    stream=False
)

# Extract and display the generated article.
kindle_nemotron_summary = (
    completion
    .choices[0]
    .message
    .content
)

print(kindle_nemotron_summary)

### Overall Customer Perception
The Kindle & Reading category earns strong overall satisfaction, with an average rating of 4.67 out of 5 across 4,679 reviews. Positive sentiment dominates at 94.9%, while neutral and negative sentiments account for 3.1% and 2.0% respectively. The data indicates that the vast majority of customers are highly satisfied with their purchase experience.

### Strengths & Use Cases
Reviewers consistently highlight the devices’ lightweight, one-handed ergonomics and multi-week battery life as core strengths. The high-resolution display and Bookerly font are praised for delivering a paper-like reading experience. The integrated Kindle store provides a vast content library that complements the hardware. Use cases include comfortable outdoor reading by the pool or deck, travel portability, and, for Fire tablet models, multitasking such as email and web browsing alongside reading.

### Complaints & Trade-offs
Several reviewers note specific limitations. Touchscreen

### Result

Nemotron successfully generated a consumer-insight article from the real
Kindle & Reading cluster using the same balanced qualitative sample and
real cluster statistics used during the previous model experiments.

The API-based model can process the complete multi-review context without
the short-context limitation encountered with T5-small.

### Decision / Finding

The real-cluster test validates the NVIDIA API workflow.

Nemotron is therefore used as the production summarization model, while
Qwen2.5-3B-Instruct remains the strongest local/open-source baseline.

Grounding constraints remain necessary because a larger model can still
produce unsupported generalizations.

### Result

Nemotron successfully generated a consumer-insight article from the real
Kindle & Reading cluster using the same balanced qualitative sample and
real cluster statistics used during the previous model experiments.

The API-based model can process the complete multi-review context without
the short-context limitation encountered with T5-small.

### Decision / Finding

The real-cluster test validates the NVIDIA API workflow.

Nemotron is therefore used as the production summarization model, while
Qwen2.5-3B-Instruct remains the strongest local/open-source baseline.

Grounding constraints remain necessary because a larger model can still
produce unsupported generalizations.

In [48]:
# Store the final production summaries.
nemotron_summaries = []

# Get the six final cluster names.
cluster_names = sorted(
    df["cluster_name"].dropna().unique()
)

# Generate one article for each cluster.
for cluster_name in cluster_names:

    print(
        f"Generating summary for: {cluster_name}"
    )

    summary = generate_cluster_summary_nemotron(
        cluster_name,
        df,
        client
    )

    nemotron_summaries.append(
        {
            "cluster_name": cluster_name,
            "summary": summary
        }
    )

    print("Done.\n")

Generating summary for: Ease of Use & Setup
Done.

Generating summary for: Fire TV & Streaming
Done.

Generating summary for: General & Mixed Feedback
Done.

Generating summary for: Kids, Family & Gifts
Done.

Generating summary for: Kindle & Reading
Done.

Generating summary for: Tablets & Value
Done.



In [49]:
# Convert the generated results into a DataFrame.
category_summaries_nemotron = pd.DataFrame(
    nemotron_summaries
)

category_summaries_nemotron

,cluster_name,summary
0,Ease of Use & Setup,### Overall Customer Perception\nCustomer sati...
1,Fire TV & Streaming,### Overall Customer Perception\nThe Fire TV &...
2,General & Mixed Feedback,### Overall Customer Perception\nThe category ...
3,"Kids, Family & Gifts","### Overall Customer Perception\nThe ""Kids, Fa..."
4,Kindle & Reading,### Overall Customer Perception\nThe Kindle & ...
5,Tablets & Value,### Overall Customer Perception\nThe tablet ca...


In [50]:
# Save the Nemotron summaries separately first.
NEMOTRON_OUTPUT_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "category_summaries_nemotron.csv"
)

category_summaries_nemotron.to_csv(
    NEMOTRON_OUTPUT_PATH,
    index=False
)

print(
    "Saved Nemotron summaries to:",
    NEMOTRON_OUTPUT_PATH
)

Saved Nemotron summaries to: /Users/caiomaia/ironhack-workspace/project-nlp-customer-reviews-df-cm/datasets/category_summaries_nemotron.csv


## 13. Final QA of Nemotron summaries

### Question

Are the six generated consumer-insight articles grounded in the review evidence?

### What we check

For each cluster, we verify that the generated article:

- reflects the real cluster statistics
- describes strengths supported by sampled reviews
- describes complaints supported by sampled reviews
- does not invent unsupported products, features or problems
- does not generalize from a single review as if it represented the whole cluster

### Decision

If all six summaries pass this qualitative review, the Nemotron output becomes the final production summarization result.

In [51]:
# Display every Nemotron summary in full
# so we can perform a final qualitative review.

for _, row in category_summaries_nemotron.iterrows():

    print("=" * 100)
    print("CLUSTER:", row["cluster_name"])
    print("=" * 100)

    print(row["summary"])

    print()

CLUSTER: Ease of Use & Setup
### Overall Customer Perception
Customer satisfaction for Ease of Use & Setup is exceptionally high. Across 2,706 reviews, the category averages 4.75 out of 5 stars, with 98.9% positive sentiment. Negative feedback accounts for only 0.3% of reviews, indicating that the vast majority of users find the experience straightforward and accessible.

### Strengths & Use Cases
Reviewers consistently describe the setup process as simple and the interface as user-friendly. The product is frequently praised for consolidating TV apps into a single location. It is highlighted as particularly suitable for children and grandchildren, who reportedly operate it without difficulty. Users also note the availability of numerous free apps and the option to expand storage via an SD card. One reviewer specifically mentioned ease of use for email and internet browsing.

### Complaints & Trade-offs
A minority of reviews identify specific friction points. One user found the reliance

In [52]:
# Build a text file containing every complete Nemotron summary.
# This makes final qualitative QA easier than using truncated notebook output.

QA_OUTPUT_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "nemotron_summaries_qa.txt"
)

# Open the file in write mode.
with open(
    QA_OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as file:

    # Go through every generated cluster summary.
    for _, row in category_summaries_nemotron.iterrows():

        # Write a clear header for the cluster.
        file.write("=" * 100 + "\n")
        file.write(
            f"CLUSTER: {row['cluster_name']}\n"
        )
        file.write("=" * 100 + "\n\n")

        # Write the complete summary.
        file.write(
            row["summary"]
        )

        # Leave space before the next cluster.
        file.write("\n\n\n")

print(
    "QA file saved to:",
    QA_OUTPUT_PATH
)


QA file saved to: /Users/caiomaia/ironhack-workspace/project-nlp-customer-reviews-df-cm/datasets/nemotron_summaries_qa.txt


## 13. Final Quality Assurance

### Question

Do the six Nemotron articles satisfy the requirements of the summarization task?

### What we check

For every generated article we verify:

- the required four sections are present
- the article is approximately 180–230 words
- real cluster statistics are used correctly
- strengths are supported by sampled reviews
- complaints are supported by sampled reviews
- the model does not invent unsupported products, features or problems
- isolated examples are not presented as widespread patterns

### Goal

Only summaries that remain grounded in the supplied reviews and statistics
should be accepted as final project outputs.

In [54]:
# Define the four sections that every final article must contain.
required_sections = [
    "### Overall Customer Perception",
    "### Strengths & Use Cases",
    "### Complaints & Trade-offs",
    "### Buyer Takeaway"
]

# Words that may indicate that the model is generalizing
# beyond what a small qualitative sample can support.
generalization_terms = [
    "frequently",
    "consistently",
    "commonly",
    "widespread",
    "typically",
    "ideal for",
    "major driver"
]

qa_results = []

# Check every generated Nemotron summary.
for _, row in category_summaries_nemotron.iterrows():

    summary = row["summary"]

    # Count the words in the generated article.
    word_count = len(summary.split())

    # Check whether every required heading exists.
    sections_present = all(
        section in summary
        for section in required_sections
    )

    # Find potentially over-generalized language.
    flagged_terms = [
        term
        for term in generalization_terms
        if term.lower() in summary.lower()
    ]

    qa_results.append({
        "cluster_name": row["cluster_name"],
        "word_count": word_count,
        "required_sections": sections_present,
        "flagged_terms": ", ".join(flagged_terms)
    })

# Convert the QA results into a DataFrame.
qa_table = pd.DataFrame(qa_results)

qa_table

,cluster_name,word_count,required_sections,flagged_terms
0,Ease of Use & Setup,263,True,"frequently, consistently, ideal for"
1,Fire TV & Streaming,284,True,frequently
2,General & Mixed Feedback,256,True,
3,"Kids, Family & Gifts",265,True,frequently
4,Kindle & Reading,293,True,"frequently, consistently"
5,Tablets & Value,270,True,consistently


In [55]:
# Print only summaries that contain language
# worth manually checking for over-generalization.

for _, row in qa_table.iterrows():

    if row["flagged_terms"]:

        cluster_name = row["cluster_name"]

        summary = category_summaries_nemotron.loc[
            category_summaries_nemotron["cluster_name"] == cluster_name,
            "summary"
        ].iloc[0]

        print("=" * 100)
        print("CLUSTER:", cluster_name)
        print("FLAGGED TERMS:", row["flagged_terms"])
        print("=" * 100)
        print(summary)
        print()

CLUSTER: Ease of Use & Setup
FLAGGED TERMS: frequently, consistently, ideal for
### Overall Customer Perception
Customer satisfaction for Ease of Use & Setup is exceptionally high. Across 2,706 reviews, the category averages 4.75 out of 5 stars, with 98.9% positive sentiment. Negative feedback accounts for only 0.3% of reviews, indicating that the vast majority of users find the experience straightforward and accessible.

### Strengths & Use Cases
Reviewers consistently describe the setup process as simple and the interface as user-friendly. The product is frequently praised for consolidating TV apps into a single location. It is highlighted as particularly suitable for children and grandchildren, who reportedly operate it without difficulty. Users also note the availability of numerous free apps and the option to expand storage via an SD card. One reviewer specifically mentioned ease of use for email and internet browsing.

### Complaints & Trade-offs
A minority of reviews identify sp

In [56]:
# Define the final production summary file.
FINAL_SUMMARY_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "category_summaries.csv"
)

# Save the validated Nemotron summaries
# as the final Task 3 output.
category_summaries_nemotron.to_csv(
    FINAL_SUMMARY_PATH,
    index=False
)

print("Final summaries saved to:")
print(FINAL_SUMMARY_PATH)

Final summaries saved to:
/Users/caiomaia/ironhack-workspace/project-nlp-customer-reviews-df-cm/datasets/category_summaries.csv


## Final Task 3 Findings

We evaluated several pretrained generative approaches for multi-review
summarization.

### Models evaluated

- T5-small
- FLAN-T5-small
- DistilBART
- Qwen2.5-3B-Instruct
- NVIDIA Nemotron 3 Ultra 550B

T5-small provided a valid local baseline but showed limited abstraction
and a short input-context constraint.

FLAN-T5-small improved instruction following but remained relatively
extractive.

DistilBART improved aggregation but lost some information.

Qwen2.5-3B-Instruct produced the strongest local/open-source results.

Following further experimentation and the deployment architecture
discussed with the instructor, NVIDIA Nemotron was selected as the
production summarization model.

### Final pipeline

    Reviews
       ↓
    KMeans cluster
       ↓
    Full-cluster statistics
       +
    Balanced qualitative sample
    (5 positive + 5 neutral + 5 negative)
       ↓
    Grounded Prompt V3
       ↓
    NVIDIA Nemotron API
       ↓
    Consumer Insight Article

The balanced sample exposes different types of feedback but is not used
to estimate sentiment prevalence. Overall sentiment statistics are
calculated from every review in the cluster.

### Limitation

Even large generative models can produce unsupported generalizations.
Prompt grounding reduces this risk but does not eliminate it completely.

Generated summaries therefore require qualitative validation before
being treated as final business insights.

### Final Decision

**NVIDIA Nemotron 3 Ultra 550B via the NVIDIA API is the production
summarization model for Task 3.**

Qwen2.5-3B-Instruct remains documented as the strongest local/open-source
baseline tested in the project.